In [1]:
# ============================================================
# BLOCK A — load, refit scorecard, score both frames
# ============================================================
import os, json
os.environ['TMP'] = os.environ['TEMP'] = r'X:\Projects\credit-pd-engine\tmp'

import pandas as pd, numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

train = pd.read_parquet('data/processed/train_binned.parquet')
test  = pd.read_parquet('data/processed/test_binned.parquet')
meta  = json.load(open('data/processed/feature_set.json'))

def woe_iv(data, feature, target='target', laplace=0.5):
    grp = data.groupby(feature, dropna=False, observed=False)[target].agg(
        n='size', bad='sum')
    grp['good'] = grp['n'] - grp['bad']
    G, B, k = grp['good'].sum(), grp['bad'].sum(), len(grp)
    grp['pct_good'] = (grp['good'] + laplace) / (G + laplace * k)
    grp['pct_bad']  = (grp['bad']  + laplace) / (B + laplace * k)
    grp['woe'] = np.log(grp['pct_good'] / grp['pct_bad'])
    grp['iv_contrib'] = (grp['pct_good'] - grp['pct_bad']) * grp['woe']
    grp['bad_rate']   = grp['bad'] / grp['n']
    grp['pct_of_pop'] = grp['n'] / grp['n'].sum()
    return (grp[['n','pct_of_pop','bad_rate','woe','iv_contrib']],
            grp['iv_contrib'].sum())

FINAL_10 = ['term','fico_bin','dti_bin','verif_grp','annual_inc_bin',
            'loan_amnt_bin','inq_last_6mths_bin','purpose_grp',
            'home_ownership_grp','emp_length_stated']
WOE_10 = [f'woe_{f}' for f in FINAL_10]

woe_maps = {f: woe_iv(train, f)[0]['woe'].to_dict() for f in FINAL_10}
for f in FINAL_10:
    for d in (train, test):
        d[f'woe_{f}'] = d[f].map(woe_maps[f])
for d in (train, test):
    d[WOE_10] = d[WOE_10].fillna(0.0)

model = sm.Logit(train['target'], sm.add_constant(train[WOE_10])).fit(disp=0)

PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)
base_pts = (offset - factor * model.params['const']) / len(WOE_10)

for d in (train, test):
    d['pd_sc'] = model.predict(sm.add_constant(d[WOE_10]))
    d['score'] = base_pts*len(WOE_10) + sum(
        -(model.params[f'woe_{f}'] * d[f'woe_{f}']) * factor for f in FINAL_10)

xgb_te = pd.read_parquet('data/processed/test_scored_xgb.parquet')
test = test.merge(xgb_te[['id','pd_xgb']], on='id', how='left')

print(f"Scorecard test AUC: {roc_auc_score(test['target'], test['pd_sc']):.4f}")
print(f"XGBoost   test AUC: {roc_auc_score(test['target'], test['pd_xgb']):.4f}")
print(f"XGB merge nulls   : {test['pd_xgb'].isna().sum()}")

Scorecard test AUC: 0.7076
XGBoost   test AUC: 0.7165
XGB merge nulls   : 0


In [ ]:
# ============================================================
# BLOCK B — characteristic PSI: has the applicant pool shifted?
# PSI = Σ (%test − %train) × ln(%test / %train)
# <0.10 stable | 0.10–0.25 moderate shift | >0.25 significant shift
# ============================================================
def psi(tr, te, feature):
    a = tr[feature].value_counts(normalize=True, dropna=False)
    b = te[feature].value_counts(normalize=True, dropna=False)
    j = pd.DataFrame({'train': a, 'test': b}).fillna(1e-6)
    j['psi'] = (j['test'] - j['train']) * np.log(j['test'] / j['train'])
    return j, j['psi'].sum()

psi_all = {}
for f in FINAL_10:
    tbl, v = psi(train, test, f)
    psi_all[f] = v
    if v > 0.10:
        print(f"\n=== {f} | PSI = {v:.4f} ===")
        print(tbl.round(4).to_string())

print("\n--- PSI ranking ---")
s = pd.Series(psi_all).sort_values(ascending=False).round(4)
print(s.to_string())
print(f"\nStable (<0.10): {(s < 0.10).sum()}/{len(s)}") 


--- PSI ranking ---
dti_bin               0.0581
inq_last_6mths_bin    0.0506
fico_bin              0.0108
verif_grp             0.0107
purpose_grp           0.0099
home_ownership_grp    0.0048
emp_length_stated     0.0037
annual_inc_bin        0.0036
loan_amnt_bin         0.0031
term                  0.0004

Stable (<0.10): 10/10


In [7]:
# ============================================================
# BLOCK C — score-level PSI, and decile lift/gains
# ============================================================
edges = np.percentile(train['score'], np.arange(0, 101, 10)).tolist()
edges[0], edges[-1] = -np.inf, np.inf
train['dec'] = pd.cut(train['score'], edges)
test['dec']  = pd.cut(test['score'],  edges)

_, psi_score = psi(train, test, 'dec')
print(f"Score PSI (train vs test): {psi_score:.4f}\n")

lift = test.groupby('dec', observed=False).agg(
    n=('target','size'), bads=('target','sum'), bad_rate=('target','mean'),
    avg_pred=('pd_sc','mean'))

lift = lift.sort_values('bad_rate', ascending=False)   # riskiest first
lift['cum_bads_%'] = (lift['bads'].cumsum() / lift['bads'].sum() * 100).round(1)
lift['cum_pop_%']  = (lift['n'].cumsum() / lift['n'].sum() * 100).round(1)
lift['lift']       = (lift['bad_rate'] / test['target'].mean()).round(2)

print("Riskiest decile first — 2015 test:")
print(lift.round(4).to_string())

Score PSI (train vs test): 0.0005

Riskiest decile first — 2015 test:
                        n   bads  bad_rate  avg_pred  cum_bads_%  cum_pop_%  lift
dec                                                                              
(-inf, 510.021]     38353  18328    0.4779    0.3841        24.2       10.2  2.37
(510.021, 520.356]  37230  12468    0.3349    0.2720        40.6       20.1  1.66
(520.356, 527.37]   39093  10047    0.2570    0.2176        53.9       30.5  1.27
(527.37, 532.83]    38606   8130    0.2106    0.1838        64.6       40.8  1.04
(532.83, 537.55]    37565   6869    0.1829    0.1591        73.7       50.8  0.91
(537.55, 542.291]   37104   5831    0.1572    0.1384        81.4       60.7  0.78
(542.291, 547.413]  37243   4906    0.1317    0.1195        87.8       70.6  0.65
(547.413, 553.601]  37302   4238    0.1136    0.1005        93.4       80.5  0.56
(553.601, 562.716]  37244   3242    0.0870    0.0799        97.7       90.5  0.43
(562.716, inf]      35806   

In [8]:
# ============================================================
# BLOCK D — where do the two models disagree?
# ============================================================
test['rank_sc']  = test['pd_sc'].rank(pct=True)
test['rank_xgb'] = test['pd_xgb'].rank(pct=True)
test['gap']      = test['rank_xgb'] - test['rank_sc']

print(f"Rank correlation (Spearman): {test[['pd_sc','pd_xgb']].corr(method='spearman').iloc[0,1]:.4f}\n")

# Where they disagree by >20 percentile points, who is right?
dis = test[test['gap'].abs() > 0.20]
print(f"Disagree by >20pp: {len(dis):,} loans ({len(dis)/len(test)*100:.1f}%)")
print(f"  XGB says riskier  — bad rate {test[test['gap'] >  0.20]['target'].mean():.3f}")
print(f"  Scorecard riskier — bad rate {test[test['gap'] < -0.20]['target'].mean():.3f}")
print(f"  Overall           — bad rate {test['target'].mean():.3f}")

Rank correlation (Spearman): 0.8827

Disagree by >20pp: 55,523 loans (14.8%)
  XGB says riskier  — bad rate 0.208
  Scorecard riskier — bad rate 0.167
  Overall           — bad rate 0.202


In [10]:
# ============================================================
# BLOCK E — walk-forward: is performance stable over time?
# Each fold trains on everything before year t, tests on year t.
# ============================================================
rows = []
for yr in [2012, 2013, 2014, 2015]:
    if yr == 2015:
        tr_f, te_f = train, test
    else:
        tr_f = train[train['vintage'].astype(int) < yr]
        te_f = train[train['vintage'].astype(int) == yr]
    if len(tr_f) < 20_000:
        continue
    maps = {f: woe_iv(tr_f, f)[0]['woe'].to_dict() for f in FINAL_10}
    Xtr = pd.DataFrame({f'woe_{f}': tr_f[f].map(maps[f]) for f in FINAL_10}).fillna(0)
    Xte = pd.DataFrame({f'woe_{f}': te_f[f].map(maps[f]) for f in FINAL_10}).fillna(0)
    m = sm.Logit(tr_f['target'].values, sm.add_constant(Xtr)).fit(disp=0)
    rows.append({'test_year': yr, 'train_n': len(tr_f), 'test_n': len(te_f),
                 'test_bad_rate': round(te_f['target'].mean(), 3),
                 'AUC': round(roc_auc_score(te_f['target'],
                              m.predict(sm.add_constant(Xte))), 4)})

wf = pd.DataFrame(rows)
print(wf.to_string(index=False))

 test_year  train_n  test_n  test_bad_rate    AUC
      2012    42506   53367          0.162 0.6712
      2013    95873  134804          0.156 0.6693
      2014   230677  223103          0.184 0.6889
      2015   453780  375546          0.202 0.7076


In [11]:
# ============================================================
# BLOCK F — Phase 5 summary
# ============================================================
summary = pd.Series({
    'scorecard_AUC':   round(roc_auc_score(test['target'], test['pd_sc']), 4),
    'xgboost_AUC':     round(roc_auc_score(test['target'], test['pd_xgb']), 4),
    'benchmark_AUC':   0.7100,
    'score_PSI':       round(psi_score, 4),
    'max_feature_PSI': round(max(psi_all.values()), 4),
    'top_decile_lift': round(lift['lift'].iloc[0], 2),
    'mean_pred_PD':    round(test['pd_sc'].mean(), 4),
    'actual_bad_rate': round(test['target'].mean(), 4),
})
print(summary.to_string())
wf.to_csv('data/processed/walkforward.csv', index=False)
lift.to_csv('data/processed/gains_table.csv')

scorecard_AUC      0.7076
xgboost_AUC        0.7165
benchmark_AUC      0.7100
score_PSI          0.0005
max_feature_PSI    0.0581
top_decile_lift    2.3700
mean_pred_PD       0.1720
actual_bad_rate    0.2019
